# Intro

This notebook is an interactive guide that you can follow along with to get to know
SQLMesh's basic concepts, the project structure, and our source data.

Let's start by getting to know our data.

## Our data source: GH Archive

The website https://www.gharchive.org/ is a free archive of all all public Github events,
commits, issues, PRs, etc.

This data can be downloaded in compressed jsons containing all events of one
hour of Github activity. For example, https://data.gharchive.org/2024-11-27-14.json.gz
contains all events from 2024-11-27 from 14:00-15:00. Each hour is about ~170 MB.

For **documentation** regarding the various event types and contents of the
corresponding jsons, see
https://docs.github.com/en/rest/using-the-rest-api/github-event-types?apiVersion=2022-11-28

DuckDB can actually read these JSONs directly using a function `READ_JSON`:

In [ ]:
import notebook_utils

# This utility function runs a query on DuckDB and displays it in this notebook:
notebook_utils.query_duckdb(
    """
    SELECT *
    FROM READ_JSON(
        'https://data.gharchive.org/2020-10-15-23.json.gz'
    )
    LIMIT 3
    """,
    attach_db=False,
)

As you can see, the columns contain jsons. Some of these jsons (e.g. `payload`) contain
lists of objects (e.g. commits) so this is not the easiest dataset to work with using SQL.

## Our SQLMesh project

Let's transform some of this data using SQLMesh. I created a SQLMesh project containing
a few models:

- **events**: The json data, ingested incrementally
- **commits**: The jsons in `payload` unrolled to get a table containing commits
- **funny_commits**: Only commits with a funny message (containing the word "stupid" or "hmm" - feel free to edit)
- **commits_stats_...**: Some statistics (totals/sums)

Here's the **lineage** as visualized by the SQLMesh VSCode extension
(you can see this by opening the `commits.sql` model and clicking the "Lineage" tab next
to the terminal)

![lineage](../docs/img/lineage.png)

## How to run the project

Like dbt, SQLMesh has a command `sqlmesh run` which does the same thing, but we can't
just use this for this project, because the data is way too large (170 MB per hour) to
process locally in codespaces.

Instead I configured the "start date" of all incremental models (see `config.yaml`)
to be 2025-01-01 00:00. You can run the project in dev and ingest 00:00 until 03:00 using

```
sqlmesh plan dev --execution-time '2025-01-01 06:00'
```

You can run the above command in the terminal. It will:

1. Show you what it intends to do (like `terraform plan`)
2. Ask you if you want to run it and if so, runs the project (like `terraform apply`)

In Jupyter notebooks you can't interactively answer prompts, but we can (approximately)
do the same with 2 individual steps:

1. Use `--explain` to do only step 1
2. Use `--auto-apply` to do step 1 & 2 (without prompting for approval)

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan dev --execution-time '2025-01-01 03:00' --explain

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan dev --execution-time '2025-01-01 03:00' --auto-apply

This should take about 30 seconds to run the project (most of it spent ingesting data).

The data is written into a local DuckDB database (`local.duckdb`). We can query it
using DuckDB:

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT * FROM persistent.gharchive__dev.funny_commits
    """
)

So far, everything is quite similar to dbt.

However, if we run `sqlmesh plan` *again*, the output is quite different:

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan dev --execution-time '2025-01-01 03:00' --auto-apply

This should print:

> No changes to plan: project files match the `dev` environment

**SQLMesh stores state:** Using a "state connection" configured in `config.yaml`
(I configured a Postgres database), SQLMesh stores the history of all the runs it has
done:

- Which models it has run & when it ran them
- For incremental models: which data intervals it has processed for each model
- A hash of the code (technically of the semantic structure of the code) of the model
  that was run

You can query this Postgres state database using the SQLTools extension (see icon on
the left), or directly via postgresql.

Let's look at e.g. the `_intervals` state table:

In [ ]:
import notebook_utils
notebook_utils.query_state_db(
    """
    SELECT *
    FROM sqlmesh._intervals
    ORDER BY name, start_ts
    """
)

Using this state database, SQLMesh knows that it has run the current code for the
current data intervals 00:00-03:00 already, so it does not need to run it again.

Note: if you ever want to start over, I made the command `make clean` which wipes both
state (Postgres) and data (DuckDB) databases.

Even more powerfully, if we create a **new environment acc** using `sqlmesh plan acc`,
sqlmesh can reuse the data from dev (since it knows that both the code and the data
intervals are the same) and it can perform a "virtual" update:

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan acc --execution-time '2025-01-01 03:00' --auto-apply

The above should be way faster than the run for firt run `dev` was, and should say:

> SKIP: No physical layer updates to perform
>
> SKIP: No model batches to execute
>
> ✔ Virtual layer updated

## How the physical layer & virtual layers work

Let's look at what the objects `gharchive__dev.commits` and `gharchive__acc.commits`
actually are in DuckDB:

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT view_definition
    FROM information_schema.views
    WHERE table_schema = 'gharchive__dev'
    AND table_name = 'commits'
    """
)

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT view_definition
    FROM information_schema.views
    WHERE table_schema = 'gharchive__acc'
    AND table_name = 'commits'
    """
)

Note that `gharchive__dev.commits` and `gharchive__acc.commits` are both **views**
pointing to the same underlying table in schema `persistent.sqlmesh__gharchive`.
This table is named `gharchive__commits__772950040` (or similar), where 772950040 can
be thought of as a "commit hash" of this model and all upstream models (because if an
upstream model changes, so should the downstream model).

## Exercises

1. Modify the code of a model (e.g. the definition of a "funny commit" in model `commits`).
2. Do a `sqlmesh plan` in dev, note how SQLMesh displays the changes you made: it shows
  a diff of your code and which models it needs to rerun.
3. Query your data in `acc`. Note that the `acc` environment is unaffected by your
  changes in dev.
4. Look at the view definition of the model you changed (or of a downstream model) in
  dev and acc. Note that these point to different tables.
5. Apply your changes in the acc environment. Note that only a virtual update is needed.
6. "Roll back" your changes in acc, e.g. do a `git stash` and a new run.
  Note again that only a virtual update is needed because the old data is still there.

For your convenience I put some convenient commands to run & query data.

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan dev --execution-time '2025-01-01 03:00' --auto-apply

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT *
    FROM persistent.gharchive__dev.commits_stats_per_hour
    ORDER BY hour ASC
    """
)

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT *
    FROM persistent.gharchive__dev.funny_commits
    ORDER BY push_timestamp
    """
)

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT table_schema, table_name, view_definition
    FROM information_schema.views
    WHERE table_schema like 'gharchive__%'
    AND table_name = 'funny_commits'
    """
)